## The first big project - The Digital Twin

### But first: introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like Agents) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [5]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [6]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

In [7]:
# Getting the Telegram bot token and chat ID from environment variables
# You can also replace these with your actual values directly

TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "your_bot_token_here")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "your_chat_id_here")

### verify TELEGRAM token and chat_id are there

if TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID:
    print("TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID found")
else:
    print("TELEGRAM_BOT_TOKEN or TELEGRAM_CHAT_ID not found")



TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID found


In [8]:
def send_telegram_message(text):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": text}
    print(f"Sending message to Telegram: {text}")
    print(f"TELEGRAM_BOT_TOKEN: {TELEGRAM_BOT_TOKEN}")
    print(f"TELEGRAM_CHAT_ID: {TELEGRAM_CHAT_ID}")
    response = requests.post(url, data=payload)

    if response.status_code == 200:
        # print("Message sent successfully!")
        return {"status": "success", "message": text}
    else:
        # print(f"Failed to send message. Status code: {response.status_code}")
        # print(response.text)
        return {"status": "error", "message": response.text}

In [10]:
# send a test messagex
send_telegram_message("Hello Telegram. This is a test message.")

Sending message to Telegram: Hello Telegram. This is a test message.
TELEGRAM_BOT_TOKEN: 8293258048:AAHVjgD8DLdWM7eG7dfgYqF8mlfOkd4QakM
TELEGRAM_CHAT_ID: 8058969323


{'status': 'success', 'message': 'Hello Telegram. This is a test message.'}

In [11]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    send_telegram_message(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [12]:
def record_unknown_question(question):
    send_telegram_message(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [13]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [14]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [15]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [16]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional info about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question'],


In [17]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

## Using Python built-in globals()

Python has a dictionary that gives us access to all global functions.

Sidenote: for sure when we deploy, we will use this in a more protected way..

In [18]:
globals()["record_unknown_question"]("this is a really hard question")

Sending message to Telegram: Recording this is a really hard question asked that I couldn't answer
TELEGRAM_BOT_TOKEN: 8293258048:AAHVjgD8DLdWM7eG7dfgYqF8mlfOkd4QakM
TELEGRAM_CHAT_ID: 8058969323


'OK'

In [19]:
# This gives us a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [20]:
reader = PdfReader("twin/bell_full.pdf")
profile = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        profile += text

with open("twin/about_me.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [21]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's profile so that you can answer questions:

{profile}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


In [23]:
from IPython.display import Markdown, display
display(Markdown(system_prompt))



# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

My name is Marcus. I love to develop digital, intelligent and smart products.
I started as an engineer and an architect, being responsible for products in the field of IoT and smart consumer products.
During my last years I shifted from being an architect more to being a Product Owner and the focus strongly shoofted into the direction of AI and products in the field of intelligent robotics.
Intelligent Semantic Worlds, Physical AI and Digital Twins - these are vocabularies you will get my attention with.

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's profile so that you can answer questions:

Marcus Bell – resume & project references 
 
MARCUS BELL 
Innovation Lead  ·  Principal Product Owner  ·  Principal Architect 
Digital Product Evolution  ·  AI & Robotics  ·  IoT & Cloud Platforms 
Niederkasseler Kirchweg 122, 40547 Düsseldorf  |  +49 176 22940533  |  marcus.bell@web.de 
 
 
P R O F E S S I O N A L  E X P E R I E N C E 
Vorwerk Group, Düsseldorf 
Innovation Lead / Principal Product Owner / Principal Architect  |  2017 – present 
End-to-end product and architecture ownership for digital ecosystems, platform services and AI -based value-
added services within the Vorwerk Group – from product vision, roadmap and backlog through to operational 
management. 
SEVEN PRINCIPLES AG (7P Solutions & Consulting), Ratingen 
Management Consultant / Team Manager Enterprise Architecture  |  2009 – 2017 
Strategic consulting and technical solution ownership in the areas of digitalization, IoT and cloud transformation. 
C-level advisory, business development and establishment of the enterprise architecture team. 
Valtech Deutschland GmbH, Düsseldorf 
Senior Consultant – Telecommunications & Telematics  |  2002 – 2009 
Enterprise architecture, solution architecture and integration management for large -scale telecommunications 
platforms – focus on Vodafone, BMW and international clients. 
Previous Positions 
ICL Deutschland GmbH, Düsseldorf – Senior Consultant eBusiness  |  2001 
Quark Deutschland GmbH, Ludwigsburg – Software Architect  |  1997 – 2001 
KW Software GmbH, Lemgo – Development Engineer / Software Architect  |  1995 – 1997 
 
Marcus Bell – resume & project references 
 
 
P R O J E C T  R E F E R E N C E S 
Vorwerk Group (2017 – present) 
01/2024 – now  |  Innovation Lead AI Services  |  Vorwerk Elektrowerke 
Design and implementation leadership for AI-based value-added services leveraging ML and AI technologies: 
LLM integration, computer vision, structure recognition, object detection, agent systems, humanoid robotic 
systems, semantic and spatial world models.  
01/2023 – 12/2023  |  Principal Product Owner Robotic Clouds  |  Vorwerk Elektrowerke 
Robot cloud migration, vendor management and change management. Stakeholder management and technical 
leadership of teams. Corporate change and organizational transformation. 
11/2017 – 12/2022  |  Principal Architect Cleaning Ecosystem  |  Vorwerk Services 
Vision, conceptualization and development of the digital ecosystem for cleaning robots and cleaning 
appliances: robot fleet management, cloud system architecture, apps and customer touchpoints. Technical 
leadership of teams, stakeholder management and partner management. 
 
SEVEN PRINCIPLES AG (2009 – 2017) 
07/2012 – 11/2017  |  Unit Manager Enterprise Architecture  |  Seven Principles AG 
Unit Manager Enterprise Architecture: responsibility for domain portfolios, team setups, CIO advisory, 
architecture management, portfolio management, Internet of Things and digitalization. 
06/2016 – 11/2017  |  Digital Architect – Digital Transformation  |  BEKO Technologies 
C-level advisory on digital transformation: formulation of vision and mission, focus and objectives. Service 
design thinking workshops and business model canvas. Development of business scenarios, definition of 
business capabilities and gap analysis. IT ro admap, initiation of measures and ongoing advisory throughout 
the transformation program. 
12/2016 – 04/2017  |  IoT Architect – Harvester Predictive Maintenance  |  Holmer Maschinenbau GmbH 
Solution design for harvester predictive maintenance. Deployment of Microsoft IoT Azure PdM, Power BI, 
Huawei IoT Platform and IoT Gateway AR 500 on the Open Telekom Cloud. Device onboarding (CAN bus, NB-
LTE), IoT platform configuration, IoT agent installation and implementation of a custom software application. 
09/2016 – 12/2016  |  IoT Architect – Multi-Cloud IoT Platform  |  T-Systems 
Platform architecture and security architecture for the Azure platform within the T -Systems IoT multi -cloud 
landscape. Identity & access management, role model, provisioning of user and access rights for service 
access. 
06/2016 – 09/2016  |  IoT Architect – IoT Showcase Innovation Lab  |  HUAWEI 
Vision, concept and development of an IoT showcase for the HUAWEI Innovation Lab: smart tracking and smart 
parking. Device onboarding with gateway and sensors (HUAWEI UBlox/SARA), connectivity (NB -LTE, WiFi, 
GPRS) and platform architecture on AWS Cloud (Lambda, DynamoDB). 
01/2016 – 06/2016  |  Agile Architect – Mobility Platform  |  Union Investment Group 
Architecture concept and evaluation of enterprise mobility platforms for a mobile app factory. Backend 
integration architecture. Concept for the agile operations and delivery model in the service lifecycle. 
01/2015 – 10/2015  |  Enterprise IoT Architect – IoT Portfolio Evolution  |  TÜV Rheinland Gruppe 
CIO advisory on service portfolio evolution in the context of digital healthcare. Use cases for wearables in 
occupational health services. Service design thinking workshops, feasibility study and service portfolio 
evolution. 
06/2014 – 10/2014  |  Enterprise Architect – BSS Evolution Roadmap  |  Telefónica Deutschland Marcus Bell – resume & project references 
 
Evolution roadmap for the BSS domain (Business Support Systems, CRM) in the context of the E -Plus and 
Telefónica merger aligned with IT strategy: digital telecommunications products, hybrid products, real -time 
provisioning, wholesale strategy and integration concept. 
05/2012 – 04/2014  |  Solution Architect – API Management  |  Vodafone Deutschland 
Design and feasibility assessment across multiple projects for various telecommunications products. 
Coordination within projects, partner management and further development of the integration domain. Projects 
(selection): Mobile Wallet, Near Field Communic ation (NFC), retail shop integration, provisioning of M2M/IoT 
devices (wearables). 
07/2010 – 11/2012  |  Enterprise Architect – Cross Domain Integration Blueprint  |  Vodafone D2 
Development of an architecture vision and domain blueprint for the target architecture of the integration domain. 
Definition of system capabilities, requirements workshops and vendor selection. Presentation of results to the 
architecture boards. 
09/2009 – 11/2015  |  Enterprise Architect – Integration Gateway Platform Evolution  |  Vodafone D2 
Development of the target architecture and guidance of platform evolution. Service governance and integration 
strategy concepts. Requirements workshops with business units, operational departments and architecture 
boards. Engineering support as lead architect. Vendor SPOC for Chinese and Indian development teams. 
05 – 09/2009  |  Enterprise Architect – BSS Domain Architecture Blueprint  |  VRS Luxemburg 
Concept and blueprint for the BSS systems architecture. Process analysis and optimization with business units. 
Requirements workshops and development of the domain architecture. 
 
Valtech Deutschland / Vodafone (2002 – 2009) 
 
2008 – 2009  |  Lead Architect – ConnectedDrive Infotainment Platform  |  BMW / ATX 
Development of the solution architecture for the infotainment platform within the ConnectedDrive program. 
Design of the architecture for generic integration of partner services (services, content providers) and creation 
of the AppStore integration concept. 
2007 – 2008  |  Lead Architect – SOHO/SME Contract Renewal Portal  |  Vodafone NL 
Architecture and high -level design for a contract renewal portal for business customers. Organization and 
facilitation of requirements workshops with business and IT departments. Implementation oversight and 
definition of acceptance criteria. 
2006 – 2007  |  Architect – POS Reference Data Optimization  |  Vodafone D2 
Business analysis of reference data management for POS applications. Optimization of the data model and 
reference data structures. Resolution of performance bottlenecks in offline tariff data systems. 
2005 – 2006  |  Integration Manager – Retail Application for Shops  |  Vodafone D2 
Integration management for a central POS application. Responsible for migration to a new product: escalation 
management, engineering team support and collaboration with offshore development departments. 
2004 – 2005  |  Architect – Common User Management  |  Vodafone D2 
Development of a generic user management system for POS applications with high security standards and a 
multi-tenant, role-based access concept for shops and partner shops. 
2003 – 2004  |  Integration Manager – Specialized Retail Online  |  Vodafone D2 
Responsible for the quality of software releases: definition of acceptance criteria, test automation and 
introduction of continuous integration. 
Earlier Project References 
2001 – 2002  |  Architect – User Care Frontend  |  ePlus / KPN 
Design and implementation of a user care frontend for call center agents to access customer data. Multi-tenant 
frontend concept with role-based access control. Marcus Bell – resume & project references 
 
1997 – 2001  |  Software Architect – eBusiness Platform Mercury  |  Quark Deutschland GmbH 
Design and implementation of modules for an eBusiness platform (customer and billing system, CRM). 
Responsible for a statistics module based on OLAP and the development of a reliable messaging pattern in the 
system backend. Management of offshore teams in India. 
1995 – 1997  |  Software Engineer – IDE for PLC Controllers  |  KW Software GmbH 
Design and implementation of a generic code generator within a complex compiler framework. Foundation for 
a series of code generators. Development of an algorithm for generating complex data structures (high -level 
languages) on controllers (binary code). 
P A T E N T S 
Self-propelled Floor Processing Device 
Development of an evaluation unit that automatically detects and registers restricted zones (no-go areas) in an 
environment map based on behavioral parameters and movement paths, or adjusts existing zones. 
Floor Cleaning Device with Floor Detection Method 
A system for cleaning floor surfaces with a control unit and electric motor. The technology utilizes induction 
voltages and return currents between stator and rotor to analyze the floor surface or to control movement 
sequences more precisely. 
E D U C A T I O N 
Diploma in Mathematics (Dipl. Math.)  |  University of Bielefeld  |  1989 – 1995 
C E R T I F I C A T I O N S  A N D  P R O F E S S I O N A L  D E V E L O P M E N T 
• TOGAF Foundation 
• Scrum Product Owner 
• Professional Scrum Master 
• PRINCE II Foundation Level 
• ITIL / Service Management 
• SAFe (Scaled Agile Framework) 
• Professional Requirements Engineering 
• Leadership Development 7P 
• Winning Complex Sales 
• Tibco Product Bootcamp 
• Unified Process Distilled Workshop 
• Design Patterns Training 
• Soft Skills Seminar “Cicero” 
• Rational Rose Training Certificate 
• XML and Java Workshop 
• Object Oriented Design with C++ 
P U B L I C A T I O N S  A N D  C O N F E R E N C E S 
• Objekt Spektrum Online (2008): “Flexible Solutions for Order Fulfilment in Telecommunications Using SOA 
Concepts” 
• SOA Congress Mainz (2007): “Flexible Solutions for Order Fulfilment in Telecommunications Using SOA 
Concepts” Marcus Bell – resume & project references 
 
• OOP Munich Vendor Track (2006): “Business Process Management in Provisioning and Order Fulfilment” 
• Valtech Whitepaper (2006): “Order Fulfilment and Provisioning Solutions” 
A V A I L A B L E  R E F E R E N C E S 
• General Higher Education Entrance Qualification (Abitur Certificate) 
• Diploma Examination Certificate and Diploma (Diploma in Mathematics) 
• Employer References: KW Software GmbH, Quark Deutschland GmbH, ICL Deutschland, Valtech 
Deutschland GmbH 

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.


In [24]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [25]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


/Users/mab/dev/agents/pingmab-agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/mab/dev/agents/pingmab-agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/mab/dev/agents/pingmab-agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: record_unknown_question
Sending message to Telegram: Recording What was Marcus's first job? asked that I couldn't answer
TELEGRAM_BOT_TOKEN: 8293258048:AAHVjgD8DLdWM7eG7dfgYqF8mlfOkd4QakM
TELEGRAM_CHAT_ID: 8058969323


/Users/mab/dev/agents/pingmab-agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/mab/dev/agents/pingmab-agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/mab/dev/agents/pingmab-agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/mab/dev/agents/pingmab-agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 

Tool called: record_unknown_question
Sending message to Telegram: Recording What is Marcus's favorite musician? asked that I couldn't answer
TELEGRAM_BOT_TOKEN: 8293258048:AAHVjgD8DLdWM7eG7dfgYqF8mlfOkd4QakM
TELEGRAM_CHAT_ID: 8058969323


/Users/mab/dev/agents/pingmab-agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Turning into Python modules

I've turned the code in the lab into python modules; that's a great practice to do after you've completed experiments in the Notebook.

You could put all the code above into 1 python script. But it's nicer to organize the code into different modules for different concerns, and that's what I've done:

`context.py` loads in the static data and constructs the System Prompt

`tools.py` contains all the code to manage and call tools, with their associated json

`app.py` contains the Gradio app and OpenAI call.

`styles.py` contains styles to apply to Gradio and this was entirely written by Claude Code!

You could have a stab at doing this yourself, then compare with my versions.

Then to try it out, open a terminal in Cursor:

`cd 1_foundations`  
`cd twin`  
`uv run app.py`

## And now for deployment

We will deploy to HuggingFace Spaces.

Before you start: remember to update the files in the `twin` directory - your LinkedIn profile and summary.txt - so that it talks about you!

Also check that there's no README file within the twin directory. If there is one, please delete it. The deploy process creates a new README file in this directory for you.

## Deployment Part 1: HuggingFace

1. Visit https://huggingface.co and set up an account  
2. From the Avatar menu on the top right, choose Access Tokens. Choose "Create New Token". Give it WRITE permissions - it needs to have WRITE permissions! Keep a record of your new key.  
3. In the Cursor Terminal, run: `uvx hf auth login --token YOUR_TOKEN_HERE`, like `uvx hf auth login --token hf_xxxxxx`, to login at the command line with your key. Afterwards, run `uvx hf auth whoami` to check you're logged in  
4. Take your new token and add it to your .env file: `HF_TOKEN=hf_xxx` for the future

## Deployment Part 2: Push!

1. Go in to the twin directory: `cd 1_foundations` then `cd twin`
2. From the twin directory, enter: `uv run gradio deploy` 
3. Follow its instructions by selecting the default values: name it `twin`, specify app.py, choose cpu-basic as the hardware, say No to needing to supply secrets, and say "no" to github actions.  

### Deployment Part 3: Secrets

1. Go to https://huggingface.co and click your Avatar, go to your profile, select the Space
2. Go to the 3 dots menu and pick Settings
3. Scroll down to Variables and Secrets section
4. Press "New Secret" (not New Variable) and enter the name of `OPENAI_API_KEY` and the value of your key from the .env file (or use the relevant key for your LLM). Be careful to get this right!
5. Repeat for `PUSHOVER_USER` and `PUSHOVER_TOKEN` from your .env file
6. Nearer the top of the settings, click "Restart space" to restart it
7. Click on App near the top to return to the app, and after it has restarted - enjoy!

### Embedding in another site

To embed this in another website, select "Embed this space" from the three-dots menu.

### Troubleshooting

If you get a gradio error, try opening the logs (the button next to the 3-dot menu).  
Try adding more debug information particularly around your keys.

### Redploying the space

Just run `uv run gradio deploy` from the twin directory. You might need to delete the file README.md that Gradio created there if you want to name your space again.

### Deleting the space

From the 3 dots menu, select the Settings screen, and there's a Delete option at the bottom.

For more information on deployment:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

### My Digital Twin

So I spend some time taking my Digital twin to the next level!  
Here it is:   
https://edwarddonner.com/avatar

Not only can you notify me with a Push, but you can chat with the real me! Here's a video with how I made it, and instructions if you want to make it too. I started with this Career Conversations app.  
https://youtu.be/srlhW4H-Gtg


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• First and foremost, deploy this for yourself! It's a real, valuable tool - the future resume..<br/>
            • Next, improve the resources - add better context about yourself. If you know RAG, then add a knowledge base about you.<br/>
            • Add in more tools! You could have a SQL database with common Q&A that the LLM could read and write from?<br/>
            • Bring in the Evaluator from the exercise in Day 4, and add other Agentic patterns.<br/>
            • Some students have added Telegram integration so that you can chat live with people on your site, along with your twin!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your resume of the future) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>